# B

let’s build this as a clean, runnable mini-module that:

- Shows that all layers live in ℝᵈ
- Shows separability emerging across layers
- Shows geometry evolving
- Sets up the idea of “latent memory = structured intermediate representations”

This will be compact, visual, and pedagogically strong.

You can paste these directly into Jupyter cells.

# 🧠 PART 0 — Imports & Setup



In [ ]:
# Cell 0: Imports

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

# 🧠 PART 1 — Nonlinear Task (XOR)

We construct a task that is not linearly separable in input space.

This is crucial because it forces the network to build a new representation space.

In [ ]:
# Cell 1: Create XOR-like dataset

N = 400

X = torch.randn(N, 2)

# Nonlinear label: XOR of signs
y = ((X[:, 0] > 0) ^ (X[:, 1] > 0)).float().unsqueeze(1)

plt.figure()
plt.scatter(X[:, 0], X[:, 1], c=y.squeeze(), cmap='bwr')
plt.title("Input Space (Not Linearly Separable)")
plt.show()

# 🧠 PART 2 — Define Small Network

We keep dimension constant across layers to mirror transformers.

Let’s use width = 2 so we can visualize directly.

In [ ]:
# Cell 2: Small MLP with constant hidden dimension

class SmallMLP(nn.Module):
    def __init__(self, d=2):
        super().__init__()
        self.l1 = nn.Linear(2, d)
        self.l2 = nn.Linear(d, d)
        self.l3 = nn.Linear(d, 1)
        self.act = nn.Tanh()
        
    def forward(self, x):
        h1 = self.act(self.l1(x))
        h2 = self.act(self.l2(h1))
        out = self.l3(h2)
        return out, h1, h2

model = SmallMLP(d=2)

# 🧠 PART 3 — Train

In [ ]:
# Cell 3: Train model

optimizer = optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.BCEWithLogitsLoss()

for epoch in range(2000):
    optimizer.zero_grad()
    logits, _, _ = model(X)
    loss = loss_fn(logits, y)
    loss.backward()
    optimizer.step()

print("Final loss:", loss.item())

# 🧠 PART 4 — Visualize Representation Spaces

This is the key conceptual demo.

In [ ]:
# Cell 4: Extract layer representations

with torch.no_grad():
    _, h1, h2 = model(X)

h1 = h1.numpy()
h2 = h2.numpy()
X_np = X.numpy()
y_np = y.numpy().squeeze()

### Plot all spaces side by side

In [ ]:
# Cell 5: Visualize geometry evolution

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(X_np[:,0], X_np[:,1], c=y_np, cmap='bwr')
axes[0].set_title("Input Space")

axes[1].scatter(h1[:,0], h1[:,1], c=y_np, cmap='bwr')
axes[1].set_title("Layer 1 Representation")

axes[2].scatter(h2[:,0], h2[:,1], c=y_np, cmap='bwr')
axes[2].set_title("Layer 2 Representation")

plt.show()

### Now pause and say:

All points are in ℝ².
But the geometry changed.
Layer 2 makes classes nearly linearly separable.

This is your “different representation spaces” moment

# 🧠 PART 5 — Probe Linear Separability

Now we quantify the idea of “features becoming linearly accessible”.

We train a linear classifier on each representation.

In [ ]:
# Cell 6: Linear probe function

def linear_probe_accuracy(features, labels):
    probe = nn.Linear(2, 1)
    optimizer = optim.Adam(probe.parameters(), lr=0.05)
    loss_fn = nn.BCEWithLogitsLoss()
    
    X_probe = torch.tensor(features, dtype=torch.float32)
    y_probe = torch.tensor(labels.reshape(-1,1), dtype=torch.float32)
    
    for _ in range(500):
        optimizer.zero_grad()
        logits = probe(X_probe)
        loss = loss_fn(logits, y_probe)
        loss.backward()
        optimizer.step()
        
    with torch.no_grad():
        preds = torch.sigmoid(probe(X_probe)) > 0.5
        acc = (preds.float() == y_probe).float().mean()
        
    return acc.item()

In [ ]:
# Cell 7: Evaluate probes

acc_input = linear_probe_accuracy(X_np, y_np)
acc_h1 = linear_probe_accuracy(h1, y_np)
acc_h2 = linear_probe_accuracy(h2, y_np)

print("Linear separability:")
print("Input space:", acc_input)
print("Layer 1:", acc_h1)
print("Layer 2:", acc_h2)

Expected result:
```
Input space: ~0.5
Layer 1: ~0.7–0.8
Layer 2: ~0.95–1.0
```
Now deliver the core statement:

- Deep networks don't change dimensionality.

They change what is linearly readable

# 🧠 PART 6 — Same Vector, Different Meaning

Now the philosophical point.

Pick a single vector from layer 2:

In [ ]:
# Cell 8: Same vector, different linear decoders

v = torch.tensor(h2[0], dtype=torch.float32)

# Two random decoders
decoder_A = torch.randn(2)
decoder_B = torch.randn(2)

print("Decoder A output:", torch.dot(decoder_A, v).item())
print("Decoder B output:", torch.dot(decoder_B, v).item())

Explain:

- The vector has no intrinsic meaning.
- Meaning emerges from the linear map applied to it.

This sets up latent memory discussion beautifully.

# 🧠 PART 7 — The Takeaway Slide (Markdown Cell)

You can close this segment with:

**What is a Representation Space?**

All layers live in the same vector space ℝᵈ.

What changes across layers is:

- The geometry
- Which features are linearly separable
- Which directions correspond to useful abstractions

A representation space is:

>> A coordinate system in which the remaining computation becomes easier.

Deep learning = progressive linearization of useful features.

**Why this works as introduction to latent memory**

Now you can pivot:

- Memory is not just stored tokens
- It is structured latent state
- Each layer reshapes the state to expose different abstractions
- Latent memory architectures manipulate these representation spaces